# AniList ratings collection

Collects anime lists from the [AniList GraphQL API](https://graphql.anilist.co) to replace the original MyAnimeList data.

**Pipeline**
1. **Seed anime**: the top `N_SEED_ANIME` anime by `SEED_SORT`.
2. **Users**: for each seed anime, up to `USERS_PER_ANIME` users who have it on their list, until `MAX_USERS` unique users are found.
3. **Manual users**: names from `EXTRA_USERNAMES` / `EXTRA_USERNAMES_FILE`. They are always fetched and marked `source = "manual"`. `ONLY_EXTRA_USERS = True` skips steps 1–2 and fetches only pending manual users.
4. **User lists**: every entry on each user's anime list.
5. **Anime metadata**: the `N_TOP_COMMON` anime rated most often in the sample, plus the top `N_TOP_RATED` seed anime.
6. **MAL synopses**: MyAnimeList's synopsis for each anime with a MAL ID, used in place of AniList's shorter descriptions. Fetched from the official MAL API when the `MAL_CLIENT_ID` environment variable is set, otherwise from [Jikan](https://jikan.moe).
7. **Assembly**: combines the checkpoints into the output files.

Every stage saves progress to `OUTPUT_DIR/checkpoints/` and resumes where it stopped. Rerunning the notebook only requests what is missing, and it can be interrupted at any point (Kernel > Interrupt) without losing more than the current chunk. `MAX_RUNTIME_MINUTES` and `MAX_USERS_THIS_RUN` limit a session. The **Progress report** cell shows where the collection stands without making requests.

**Sampling bias**
- Users are found through top anime, so the sample leans towards active, engaged users with long lists, not typical AniList users.
- AniList only returns the first 5,000 list entries for an anime, oldest first, and ignores sorts and date filters on this query. Users therefore come from the *earliest* entries of each seed anime. For popular anime these are accounts from AniList's first years. `USER_PAGE_SAMPLING = "random"` spreads picks across that window but cannot reach past it. Less popular seed anime reach further.
- Private lists are skipped, and `SEED_REQUIRE_SCORED` keeps only users who scored the seed anime.
- Manual users are hand-picked, not a random sample. Exclude them (`source == "manual"`) from any statistics about the user population.

**Before publishing anything**: the raw output contains usernames, AniList user IDs and complete public lists. Check the terms of use in [AniList's API docs](https://docs.anilist.co/) before publishing any collected data, and only commit anonymised data. `python scripts/anonymise_users.py` writes an anonymised copy to `data/raw/anilist_anonymised/`: seed users get random IDs, while manual users keep their AniList username, since they are people of interest. Even with IDs replaced, a full list with dates can often be matched back to a public profile.

## Output files

Written to `OUTPUT_DIR` (default `data/raw/anilist/`, gitignored) by the assembly stage:

| File | Contents |
|---|---|
| `ratings.csv.gz` | One row per list entry: `anilist_user_id`, `username`, `source`, `anime_id`, `anime_title`, `status`, `score` (in `SCORE_FORMAT`), `rating` (score rescaled to 1–10, empty if unrated), `is_rated`, `progress`, `repeat`, `started_at`, `completed_at`, `updated_at`, `created_at` |
| `anime_metadata.csv` | One row per anime. `anime_ID` is the AniList ID and `mal_ID` the MAL ID. `Synopsis` is MAL's synopsis where there is one, otherwise AniList's (`synopsis_source` says which; AniList's is also kept in `synopsis_anilist`). See the README for how each column is derived |
| `user_profiles.csv` | One row per user, computed from their list: status counts, rewatches, episodes, mean score. `Gender` is always empty (AniList doesn't expose it), and `Days` is empty when AniList has no watch-time statistics |
| `users.csv` | Every username found, with `source` and `status` (`done`, `private`, `not_found`, `failed`) |

Ratings are in long format rather than a wide user × anime matrix, which with ~10,000 users and 20,000 anime would be hundreds of millions of mostly empty cells.

`anime_id` (ratings) and `anime_ID` (metadata) are AniList media IDs. `anime_title` is the romaji title, and is only filled for anime with metadata.

## Configuration

In [1]:
from pathlib import Path
import time

TEST_MODE = False                    # small end-to-end run in its own folder; set False for the real collection

# Seed anime and user discovery
N_SEED_ANIME = 500
SEED_SORT = "SCORE_DESC"            # or "POPULARITY_DESC"
USERS_PER_ANIME = 50
USER_PAGE_SAMPLING = "random"       # "random" pages within AniList's 5,000-entry depth limit, or "first"
MAX_PAGES_PER_ANIME = 3             # pages of 50 entries tried per seed anime
SEED_REQUIRE_SCORED = True          # only take users who scored the seed anime
MAX_USERS = 1000

# Manually added users
EXTRA_USERNAMES = ["Kjarr"]
EXTRA_USERNAMES_FILE = None         # optional text file, one username per line ("#" lines are ignored)
ONLY_EXTRA_USERS = False            # skip seed anime and user discovery; only fetch pending manual users

# User lists
SCORE_FORMAT = "POINT_100"          # POINT_100, POINT_10_DECIMAL, POINT_10, POINT_5 or POINT_3
DROP_UNRATED = False                # True drops score-0 (unrated) entries from ratings.csv.gz

# Anime metadata
N_TOP_COMMON = 10_500
N_TOP_RATED = 100
THEME_MIN_RANK = 60                 # tags with at least this relevance (0-100) go into Theme(s)

# MAL synopses (looked up by each anime's MAL ID)
FETCH_MAL_SYNOPSES = True
MAL_SOURCE = "auto"                 # "mal_api" (needs the MAL_CLIENT_ID environment variable), "jikan" (no key), or "auto": mal_api when MAL_CLIENT_ID is set
MAL_REQUESTS_PER_MINUTE = 50        # Jikan allows 60/min and 3/s

# Requests
REQUESTS_PER_MINUTE = 45            # AniList is degraded to 30/min (normally 90)
MAX_RETRIES = 6                     # for 5xx and network errors

# Output and checkpoints
OUTPUT_DIR = Path("data/raw/anilist")
RESUME = True                       # False backs up the checkpoints (after confirmation) and starts over
CHECKPOINT_EVERY = 100              # users per ratings checkpoint file
MAX_RUNTIME_MINUTES = None          # stop cleanly after this long (None = no limit)
MAX_USERS_THIS_RUN = None           # stop cleanly after fetching this many lists (None = no limit)
RETRY_FAILED = False                # retry users whose fetch failed after MAX_RETRIES
ASSEMBLE_PARTIAL = False            # allow metadata and assembly before every user is done
RANDOM_SEED = 42

if TEST_MODE:
    N_SEED_ANIME = 5
    USERS_PER_ANIME = 10
    MAX_PAGES_PER_ANIME = 2
    MAX_USERS = 20
    N_TOP_COMMON = 50
    N_TOP_RATED = 5
    CHECKPOINT_EVERY = 5
    OUTPUT_DIR = Path("data/raw/anilist_test")

SESSION_START = time.monotonic()    # MAX_RUNTIME_MINUTES counts from here; rerun this cell to reset it

## Setup

In [2]:
import gzip
import html
import json
import math
import os
import random
import re
import subprocess
from collections import Counter
from datetime import datetime, timezone

import pandas as pd
import requests
from tqdm import tqdm

API_URL = "https://graphql.anilist.co"
PER_PAGE = 50                       # AniList's maximum page size
MAX_PAGE_DEPTH = 5_000              # AniList rejects pages past the first 5,000 entries of a query
SCORE_MAX = {"POINT_100": 100, "POINT_10_DECIMAL": 10, "POINT_10": 10, "POINT_5": 5, "POINT_3": 3}
USER_STATUSES = {"pending", "done", "private", "not_found", "failed"}

CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
SEED_PATH = CHECKPOINT_DIR / "seed_anime.json"
USERNAMES_PATH = CHECKPOINT_DIR / "usernames.csv"
RATINGS_PARTS_DIR = CHECKPOINT_DIR / "ratings_parts"
METADATA_PARTS_DIR = CHECKPOINT_DIR / "metadata_parts"
METADATA_DONE_PATH = CHECKPOINT_DIR / "metadata_done.json"
MAL_SYNOPSIS_PARTS_DIR = CHECKPOINT_DIR / "mal_synopsis_parts"
LOG_PATH = OUTPUT_DIR / "collection.log"

assert SEED_SORT in ("SCORE_DESC", "POPULARITY_DESC")
assert SCORE_FORMAT in SCORE_MAX
assert USER_PAGE_SAMPLING in ("random", "first")
assert MAL_SOURCE in ("auto", "mal_api", "jikan")
MAL_CLIENT_ID = os.environ.get("MAL_CLIENT_ID")  # read from the environment so it never ends up in the notebook


def now_iso():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def log(message):
    """Append to the collection log. It can contain usernames, so it stays in the gitignored OUTPUT_DIR."""
    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(f"{now_iso()} {message}\n")


def atomic_write(path, write):
    """Write through a temp file and rename it into place, so an interrupt can't leave a half-written file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".tmp_{path.name}")
    write(tmp)
    os.replace(tmp, path)


def write_json(path, obj):
    atomic_write(path, lambda tmp: tmp.write_text(json.dumps(obj, indent=1), encoding="utf-8"))


def read_json(path, default):
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else default


def write_csv(df, path):
    atomic_write(path, lambda tmp: df.to_csv(tmp, index=False))


def next_part_path(folder, suffix):
    """The next unused part_NNNNN file name, so checkpoint parts are never overwritten."""
    folder.mkdir(parents=True, exist_ok=True)
    numbers = [int(m.group(1)) for p in folder.glob(f"part_*{suffix}") if (m := re.match(r"part_(\d+)", p.name))]
    return folder / f"part_{max(numbers, default=0) + 1:05d}{suffix}"


def out_of_time():
    return MAX_RUNTIME_MINUTES is not None and time.monotonic() - SESSION_START >= MAX_RUNTIME_MINUTES * 60


def hide_usernames(df):
    """Drop usernames and AniList user IDs before displaying a table; either one identifies a profile."""
    return df.drop(columns=[c for c in ("username", "username_key", "user", "anilist_user_id") if c in df.columns])


def check_gitignored():
    """Warn loudly if OUTPUT_DIR is not gitignored, since it holds usernames."""
    probe = OUTPUT_DIR / "ratings.csv.gz"
    try:
        ignored = subprocess.run(["git", "check-ignore", "-q", probe.as_posix()], capture_output=True).returncode == 0
    except FileNotFoundError:
        ignored = False
    if ignored:
        print(f"{OUTPUT_DIR.as_posix()}/ is gitignored")
    else:
        print(f"WARNING: {OUTPUT_DIR.as_posix()}/ is NOT gitignored. It will contain usernames; add it to .gitignore before committing.")


def prepare_checkpoint_dir():
    """Create the checkpoint folders. With RESUME = False, back up existing checkpoints first, after confirmation."""
    if not RESUME and CHECKPOINT_DIR.exists() and any(CHECKPOINT_DIR.iterdir()):
        backup = CHECKPOINT_DIR.with_name(f"checkpoints_backup_{datetime.now():%Y%m%d_%H%M%S}")
        try:
            answer = input(f"RESUME = False: move {CHECKPOINT_DIR.as_posix()} to {backup.name} and start over? Type 'yes': ")
        except Exception:  # no stdin, e.g. when run with nbconvert
            answer = ""
        if answer.strip().lower() != "yes":
            raise RuntimeError("Not confirmed, nothing was changed. Set RESUME = True to continue the existing collection.")
        CHECKPOINT_DIR.rename(backup)
        print(f"Old checkpoints moved to {backup.as_posix()}. Set RESUME = True again before the next run.")
    for folder in (CHECKPOINT_DIR, RATINGS_PARTS_DIR, METADATA_PARTS_DIR, MAL_SYNOPSIS_PARTS_DIR):
        folder.mkdir(parents=True, exist_ok=True)


check_gitignored()
prepare_checkpoint_dir()
print(f"Checkpoints: {CHECKPOINT_DIR.as_posix()} (resume = {RESUME}, test mode = {TEST_MODE})")

data/raw/anilist/ is gitignored
Checkpoints: data/raw/anilist/checkpoints (resume = True, test mode = False)


### Request helpers

Shared by AniList and MAL: an even-spacing throttle, `Retry-After` handling and exponential backoff.

In [3]:
class AniListError(Exception):
    """A request AniList rejected. Not retried."""


class UserNotFound(AniListError):
    pass


class PrivateUser(AniListError):
    pass


class RetriesExhausted(Exception):
    """Network or server errors persisted after MAX_RETRIES retries."""


class Throttle:
    """Spaces requests evenly. The rate is passed on each call, so changing it in the config takes effect on rerun."""

    def __init__(self):
        self.next_request_at = 0.0

    def wait(self, requests_per_minute):
        now = time.monotonic()
        if now < self.next_request_at:
            time.sleep(self.next_request_at - now)
        self.next_request_at = max(now, self.next_request_at) + 60 / requests_per_minute


def retry_after_seconds(response, default):
    try:
        return int(response.headers.get("Retry-After", default))
    except ValueError:
        return default


def back_off(failures, problem, max_delay, label=""):
    """Sleep before the next retry, or raise RetriesExhausted once MAX_RETRIES is used up."""
    if failures > MAX_RETRIES:
        raise RetriesExhausted(problem)
    delay = min(2 ** failures, max_delay) + random.random()
    log(f"{label}{problem}; retry {failures}/{MAX_RETRIES} in {delay:.0f}s")
    time.sleep(delay)


SESSION = requests.Session()
SESSION.headers.update({"Content-Type": "application/json", "Accept": "application/json"})
REQUEST_STATS = Counter()
ANILIST_THROTTLE = Throttle()   # AniList's headers may not show a degraded limit, so it's enforced here


def post_graphql(query, variables=None):
    """
    POST a GraphQL query and return its `data`.

    HTTP 429 waits for Retry-After and tries again. 5xx and network errors back off exponentially,
    up to MAX_RETRIES retries, then raise RetriesExhausted. Other errors raise AniListError
    (UserNotFound / PrivateUser for user lookups).
    """
    failures = 0
    while True:
        ANILIST_THROTTLE.wait(REQUESTS_PER_MINUTE)
        REQUEST_STATS["requests"] += 1
        try:
            response = SESSION.post(API_URL, json={"query": query, "variables": variables or {}}, timeout=60)
        except requests.RequestException as exc:
            problem = f"{type(exc).__name__}: {exc}"
        else:
            if response.status_code == 429:
                wait = retry_after_seconds(response, default=60)
                REQUEST_STATS["rate_limited"] += 1
                log(f"HTTP 429, waiting {wait}s")
                time.sleep(wait + 1)
                continue
            try:
                payload = response.json()
            except ValueError:
                payload = None
            if response.status_code < 500 and payload is not None:
                errors = payload.get("errors") or []
                if not errors:
                    return payload["data"]
                messages = "; ".join(e.get("message", "") for e in errors)
                if "Private User" in messages:
                    raise PrivateUser(messages)
                if "User not found" in messages:
                    raise UserNotFound(messages)
                raise AniListError(f"HTTP {response.status_code}: {messages}")
            problem = f"HTTP {response.status_code}"

        failures += 1
        REQUEST_STATS["retries"] += 1
        back_off(failures, problem, max_delay=120)

### Queries

In [4]:
SEED_QUERY = """
query ($page: Int, $perPage: Int, $sort: [MediaSort]) {
  Page(page: $page, perPage: $perPage) {
    pageInfo { hasNextPage }
    media(type: ANIME, sort: $sort) { id idMal popularity title { romaji } }
  }
}
"""

# Sorting and date filters return nothing on this query, and only the first 5,000 entries
# (oldest first) can be paged through
USERS_QUERY = """
query ($mediaId: Int, $page: Int, $perPage: Int) {
  Page(page: $page, perPage: $perPage) {
    mediaList(mediaId: $mediaId) { score status user { id name } }
  }
}
"""

_LIST_FIELDS = """
    hasNextChunk
    user { id name statistics { anime { count minutesWatched } } }
    lists {
      isCustomList
      entries {
        mediaId status score(format: $scoreFormat) progress repeat
        startedAt { year month day } completedAt { year month day } updatedAt createdAt
      }
    }
"""

# Separate queries by ID and by name: sending userName: null alongside userId makes AniList return HTTP 500
LIST_BY_ID_QUERY = """
query ($userId: Int, $scoreFormat: ScoreFormat) {
  MediaListCollection(userId: $userId, type: ANIME) {""" + _LIST_FIELDS + """  }
}
"""

LIST_BY_NAME_QUERY = """
query ($userName: String, $scoreFormat: ScoreFormat) {
  MediaListCollection(userName: $userName, type: ANIME) {""" + _LIST_FIELDS + """  }
}
"""

LIST_CHUNK_QUERY = """
query ($userId: Int, $scoreFormat: ScoreFormat, $chunk: Int, $perChunk: Int) {
  MediaListCollection(userId: $userId, type: ANIME, chunk: $chunk, perChunk: $perChunk) {""" + _LIST_FIELDS + """  }
}
"""

METADATA_QUERY = """
query ($ids: [Int], $perPage: Int) {
  Page(page: 1, perPage: $perPage) {
    media(id_in: $ids, type: ANIME) {
      id idMal title { romaji english native }
      format episodes duration status season seasonYear
      startDate { year month day } endDate { year month day }
      genres tags { name rank category isMediaSpoiler }
      studios(isMain: true) { nodes { name } }
      source averageScore meanScore popularity favourites isAdult
      rankings { rank type allTime }
      stats { scoreDistribution { score amount } }
      description(asHtml: false)
    }
  }
}
"""

### Checkpoint helpers

In [5]:
USERNAME_COLUMNS = ["username", "username_key", "anilist_user_id", "source", "status", "found_via_anime_id",
                    "added_at", "fetched_at", "n_entries", "episodes_watched", "minutes_watched", "error"]
NUMERIC_USER_COLUMNS = ["anilist_user_id", "found_via_anime_id", "n_entries", "episodes_watched", "minutes_watched"]
PART_COLUMNS = ["anilist_user_id", "anime_id", "status", "score", "progress", "repeat",
                "started_at", "completed_at", "updated_at", "created_at"]
MAL_SYNOPSIS_COLUMNS = ["mal_id", "synopsis", "status", "fetched_at"]


# --- Users -------------------------------------------------------------------------------------

def load_usernames():
    if not USERNAMES_PATH.exists():
        return pd.DataFrame({c: pd.Series(dtype="Int64" if c in NUMERIC_USER_COLUMNS else "object") for c in USERNAME_COLUMNS})
    # keep_default_na=False so usernames such as "NA" or "null" stay strings
    df = pd.read_csv(USERNAMES_PATH, dtype=str, keep_default_na=False).astype(object)
    for c in NUMERIC_USER_COLUMNS:
        df[c] = pd.to_numeric(df[c].where(df[c] != ""), errors="coerce").round().astype("Int64")
    return df


def save_usernames(usernames):
    write_csv(usernames[USERNAME_COLUMNS], USERNAMES_PATH)


def users_complete(usernames):
    return not usernames["status"].isin(["pending", "failed"]).any()


# --- List entries ------------------------------------------------------------------------------

def latest_parts():
    """The ratings part files, and for each user the part holding their most recent fetch."""
    parts = sorted(RATINGS_PARTS_DIR.glob("part_*.csv.gz"))
    latest = {}
    for i, path in enumerate(parts):
        for user_id in pd.read_csv(path, usecols=["anilist_user_id"])["anilist_user_id"].unique():
            latest[user_id] = i
    return parts, latest


def drop_duplicate_entries(df):
    """
    Keep one entry per user and anime.

    AniList sometimes holds two separate list entries for the same anime (mostly on accounts from
    2014-2015, e.g. one under Watching and one under Completed). Keep the rated one, then the most
    recently updated, then the one with the most progress.
    """
    if not df.duplicated(["anilist_user_id", "anime_id"]).any():
        return df
    keys = ["_rated"] + [c for c in ("updated_at", "progress") if c in df.columns]
    ranked = df.assign(_rated=df["score"] > 0 if "score" in df.columns else False)
    ranked = ranked.sort_values(keys, ascending=False, na_position="last", kind="stable")
    return ranked.drop_duplicates(["anilist_user_id", "anime_id"]).drop(columns="_rated").sort_index()


def iter_latest_entries(usecols=None, dedupe=True):
    """
    Each part's entries, keeping only every user's most recent fetch, and one entry per anime.

    A user is written whole within one part, so a user fetched twice (e.g. re-fetched after an
    interrupt before their status was saved) appears in two parts. Keeping only the latest fetch
    removes those duplicate user/anime pairs, and also drops entries the user has since deleted.
    """
    parts, latest = latest_parts()
    for i, path in enumerate(parts):
        df = pd.read_csv(path, usecols=usecols)
        df = df[df["anilist_user_id"].map(latest) == i]
        yield drop_duplicate_entries(df) if dedupe else df


def rated_counts():
    """Number of users in the sample who scored each anime."""
    total = pd.Series(dtype="float64")
    for df in iter_latest_entries(["anilist_user_id", "anime_id", "score"]):
        total = total.add(df.loc[df["score"] > 0, "anime_id"].value_counts(), fill_value=0)
    return total.astype("int64")


# --- Anime metadata ----------------------------------------------------------------------------

def metadata_targets(seed_state):
    """The N_TOP_COMMON anime scored most often in the sample, plus the top N_TOP_RATED seed anime."""
    counts = rated_counts()
    ranked = counts.rename_axis("anime_id").reset_index(name="n").sort_values(["n", "anime_id"], ascending=[False, True])
    top_common = ranked["anime_id"].head(N_TOP_COMMON).tolist()
    top_rated = [a["id"] for a in (seed_state or {}).get("anime", [])[:N_TOP_RATED]]
    return list(dict.fromkeys(int(i) for i in top_common + top_rated)), counts


def load_metadata_progress():
    """Anime IDs whose metadata was fetched ("done") or that AniList didn't return ("missing")."""
    return read_json(METADATA_DONE_PATH, {"done": [], "missing": []})


def metadata_finished():
    progress = load_metadata_progress()
    return set(progress["done"]) | set(progress["missing"])


def metadata_records():
    """Raw AniList metadata by anime ID, from the checkpoint parts (a batch fetched twice keeps the later copy)."""
    records = {}
    for path in sorted(METADATA_PARTS_DIR.glob("part_*.jsonl")):
        for line in path.read_text(encoding="utf-8").splitlines():
            m = json.loads(line)
            records[m["id"]] = m
    return records


# --- MAL synopses ------------------------------------------------------------------------------

def mal_ids_with_metadata():
    """MAL IDs of every anime with fetched metadata."""
    return {m["idMal"] for m in metadata_records().values() if m["idMal"]}


def load_mal_synopses():
    """MAL IDs already looked up (status found / empty / not_found), latest lookup per ID."""
    parts = sorted(MAL_SYNOPSIS_PARTS_DIR.glob("part_*.csv"))
    if not parts:
        return pd.DataFrame(columns=MAL_SYNOPSIS_COLUMNS)
    df = pd.concat([pd.read_csv(p, dtype={"synopsis": str}, keep_default_na=False) for p in parts], ignore_index=True)
    return df.drop_duplicates("mal_id", keep="last").reset_index(drop=True)


def mal_synopses_left():
    """MAL IDs with metadata that haven't been looked up yet."""
    return mal_ids_with_metadata() - set(load_mal_synopses()["mal_id"])


# --- Validation --------------------------------------------------------------------------------

class Check:
    """Collects validation results and fails the cell if any check failed."""

    def __init__(self):
        self.failed = []

    def __call__(self, ok, message):
        print(f"[{'PASS' if ok else 'FAIL'}] {message}")
        if not ok:
            self.failed.append(message)

    def done(self):
        if self.failed:
            raise AssertionError(f"{len(self.failed)} check(s) failed: {self.failed}")

## Progress report

Reads the checkpoints only and makes no requests. Safe to run at any time, including on its own after the setup cells.

In [6]:
def estimate_requests_left(seed, users, meta_progress):
    """Approximate requests left per stage: discovery (~1-2 pages per seed anime), one per user list, one per 50 anime of metadata."""
    status = users["status"].value_counts()
    n_seed_users = int((users["source"] == "seed").sum())
    pending_seed = [a for a in seed["anime"][:N_SEED_ANIME] if not a["users_collected"]] if seed else []

    to_discover = 0 if ONLY_EXTRA_USERS else max(MAX_USERS - n_seed_users, 0)
    discovery = len(pending_seed) * min(MAX_PAGES_PER_ANIME, 2) if to_discover else 0
    if not seed and not ONLY_EXTRA_USERS:
        discovery += math.ceil(max(N_SEED_ANIME, N_TOP_RATED) / PER_PAGE)

    lists = int(status.get("pending", 0)) + (int(status.get("failed", 0)) if RETRY_FAILED else 0) + to_discover

    finished = set(meta_progress["done"]) | set(meta_progress["missing"])
    if len(users) and users_complete(users):
        # Exact once every user is done: the targets come from the checkpoints (no requests)
        targets, _ = metadata_targets(seed)
        n_meta_left = len(set(targets) - finished)
    else:
        n_meta_left = max(N_TOP_COMMON + N_TOP_RATED - len(finished), 0)
    metadata = math.ceil(n_meta_left / PER_PAGE)

    return {"discovery": discovery, "lists": lists, "to_discover": to_discover, "metadata": metadata}


def progress_report():
    seed = read_json(SEED_PATH, None)
    users = load_usernames()
    status = users["status"].value_counts()
    meta_progress = load_metadata_progress()

    print(f"Output folder: {OUTPUT_DIR.as_posix()}")
    if seed:
        n_seed = min(len(seed["anime"]), N_SEED_ANIME)
        n_searched = sum(a["users_collected"] for a in seed["anime"][:N_SEED_ANIME])
        print(f"Seed anime: {len(seed['anime'])} saved ({seed['sort']}), {n_searched} of {n_seed} searched for users")
    else:
        print("Seed anime: not fetched yet")

    n_seed_users = int((users["source"] == "seed").sum())
    n_manual = int((users["source"] == "manual").sum())
    print(f"Users: {len(users)} listed ({n_seed_users} seed of MAX_USERS {MAX_USERS}, {n_manual} manual)")
    for s in ["done", "pending", "failed", "private", "not_found"]:
        print(f"  {s:<10}{int(status.get(s, 0))}")

    print(f"Ratings checkpoint files: {len(list(RATINGS_PARTS_DIR.glob('part_*.csv.gz')))}")
    print(f"Anime metadata: {len(meta_progress['done'])} fetched, {len(meta_progress['missing'])} not returned by AniList")

    left = estimate_requests_left(seed, users, meta_progress)
    total = left["discovery"] + left["lists"] + left["metadata"]
    not_found_yet = f" incl. {left['to_discover']:,} users not yet found" if left["to_discover"] else ""
    print(f"Estimated requests left: ~{total:,} ({left['discovery']:,} discovery, {left['lists']:,} user lists{not_found_yet}, "
          f"{left['metadata']:,} metadata) = ~{total / REQUESTS_PER_MINUTE / 60:.1f} h at {REQUESTS_PER_MINUTE}/min")

    if FETCH_MAL_SYNOPSES and meta_progress["done"]:
        n_mal, n_mal_left = len(mal_ids_with_metadata()), len(mal_synopses_left())
        print(f"MAL synopses: {n_mal - n_mal_left} of {n_mal} looked up; "
              f"~{n_mal_left / MAL_REQUESTS_PER_MINUTE / 60:.1f} h left at {MAL_REQUESTS_PER_MINUTE}/min")

    files = [p for p in CHECKPOINT_DIR.rglob("*") if p.is_file()]
    if files:
        print(f"Last checkpoint: {datetime.fromtimestamp(max(p.stat().st_mtime for p in files)):%Y-%m-%d %H:%M:%S}")


progress_report()

Output folder: data/raw/anilist
Seed anime: 500 saved (SCORE_DESC), 282 of 500 searched for users
Users: 10001 listed (10000 seed of MAX_USERS 1000, 1 manual)
  done      9483
  pending   0
  failed    0
  private   518
  not_found 0
Ratings checkpoint files: 103
Anime metadata: 15000 fetched, 0 not returned by AniList
Estimated requests left: ~0 (0 discovery, 0 user lists, 0 metadata) = ~0.0 h at 45/min
MAL synopses: 14660 of 14660 looked up; ~0.0 h left at 50/min
Last checkpoint: 2026-10-15 11:38:35


## 1. Seed anime

In [7]:
def fetch_seed_anime():
    """The top max(N_SEED_ANIME, N_TOP_RATED) anime by SEED_SORT, saved to seed_anime.json."""
    n_needed = max(N_SEED_ANIME, N_TOP_RATED)
    state = read_json(SEED_PATH, None)
    if state and state["sort"] != SEED_SORT:
        print(f"seed_anime.json was built with {state['sort']}; keeping it. Set RESUME = False to start over with {SEED_SORT}.")
        return state
    if state and len(state["anime"]) >= n_needed:
        print(f"Seed anime: {len(state['anime'])} already saved, no requests needed")
        return state

    previous = {a["id"]: a for a in (state or {}).get("anime", [])}  # keep progress flags if extending the list
    start = REQUEST_STATS["requests"]
    media = []
    for page in tqdm(range(1, math.ceil(n_needed / PER_PAGE) + 1), desc="Seed anime pages"):
        data = post_graphql(SEED_QUERY, {"page": page, "perPage": PER_PAGE, "sort": [SEED_SORT]})
        media += data["Page"]["media"]
        if not data["Page"]["pageInfo"]["hasNextPage"]:
            break

    anime = []
    for rank, m in enumerate(media[:n_needed], start=1):
        before = previous.get(m["id"], {})
        anime.append({
            "id": m["id"],
            "id_mal": m["idMal"],
            "title": m["title"]["romaji"],
            "popularity": m["popularity"],
            "seed_rank": rank,
            "users_collected": before.get("users_collected", False),
            "users_found": before.get("users_found", 0),
        })
    state = {"sort": SEED_SORT, "updated_at": now_iso(), "anime": anime}
    write_json(SEED_PATH, state)
    print(f"Seed anime: {len(anime)} saved ({REQUEST_STATS['requests'] - start} requests)")
    return state


if ONLY_EXTRA_USERS:
    seed_state = read_json(SEED_PATH, None)
    print("ONLY_EXTRA_USERS: skipping seed anime")
else:
    seed_state = fetch_seed_anime()

Seed anime: 500 already saved, no requests needed


In [8]:
# Validation: seed anime
check = Check()
if seed_state is None:
    print("No seed anime saved (ONLY_EXTRA_USERS)")
else:
    seed_df = pd.DataFrame(seed_state["anime"])
    check(len(seed_df) >= max(N_SEED_ANIME, N_TOP_RATED), f"{len(seed_df)} seed anime saved (need {max(N_SEED_ANIME, N_TOP_RATED)})")
    check(seed_df["id"].notna().all(), "no missing anime IDs")
    check(not seed_df["id"].duplicated().any(), "no duplicate anime IDs")
    check(seed_df["seed_rank"].tolist() == list(range(1, len(seed_df) + 1)), "seed ranks run 1..n")
    print(f"{seed_df['id_mal'].notna().mean():.0%} have a MAL ID")
    display(seed_df.head(10))
check.done()

[PASS] 500 seed anime saved (need 500)
[PASS] no missing anime IDs
[PASS] no duplicate anime IDs
[PASS] seed ranks run 1..n
99% have a MAL ID


,id,id_mal,title,popularity,seed_rank,users_collected,users_found
0,154587,52991.0,Sousou no Frieren,489460,1,True,50
1,114129,39486.0,Gintama: THE FINAL,56353,2,True,30
2,20996,28977.0,Gintama°,120558,3,True,50
3,189046,61316.0,Re:Zero kara Hajimeru Isekai Seikatsu 4th Season,150382,4,True,50
4,171627,57555.0,Chainsaw Man: Reze-hen,245330,5,True,50
5,5114,5114.0,Hagane no Renkinjutsushi: FULLMETAL ALCHEMIST,736448,6,True,50
6,104578,38524.0,Shingeki no Kyojin Season 3 Part 2,617120,7,True,0
7,185874,60636.0,BLEACH: Sennen Kessen-hen - Kashin-tan,92723,8,True,19
8,182469,60022.0,ONE PIECE FAN LETTER,59332,9,True,50
9,124194,42938.0,Fruits Basket: The Final,178404,10,True,50


## 2. Collect usernames

Manual users from `EXTRA_USERNAMES` / `EXTRA_USERNAMES_FILE` are registered first, on every run, so names added later are picked up as `pending`.

In [9]:
def read_extra_usernames():
    """EXTRA_USERNAMES plus EXTRA_USERNAMES_FILE, without blanks, comments or case-insensitive repeats."""
    names = list(EXTRA_USERNAMES)
    if EXTRA_USERNAMES_FILE:
        path = Path(EXTRA_USERNAMES_FILE)
        if path.exists():
            names += path.read_text(encoding="utf-8").splitlines()
        else:
            print(f"EXTRA_USERNAMES_FILE not found: {path}")
    unique, seen = [], set()
    for name in names:
        name = name.strip()
        if name and not name.startswith("#") and name.lower() not in seen:
            seen.add(name.lower())
            unique.append(name)
    return unique


def register_manual_users(usernames):
    """Add new manual usernames as pending. Matching is case-insensitive; a user already listed keeps their row."""
    extra = read_extra_usernames()
    known = set(usernames["username_key"])
    new = [name for name in extra if name.lower() not in known]
    if new:
        rows = pd.DataFrame({"username": new, "username_key": [n.lower() for n in new],
                             "source": "manual", "status": "pending", "added_at": now_iso()})
        usernames = pd.concat([usernames, rows], ignore_index=True)
        save_usernames(usernames)
    print(f"Manual users: {len(extra)} requested, {len(new)} newly added as pending, {len(extra) - len(new)} already listed")
    if new:
        print("  added:", ", ".join(new))
    return usernames


def candidate_pages(media):
    """Pages of the anime's user list to try: random pages within the reachable window, or the first pages."""
    n_pages = max(1, math.ceil(min(media["popularity"] or 0, MAX_PAGE_DEPTH) / PER_PAGE))
    n_try = min(MAX_PAGES_PER_ANIME, n_pages)
    if USER_PAGE_SAMPLING == "first":
        return list(range(1, n_try + 1))
    return random.Random(f"{RANDOM_SEED}-{media['id']}").sample(range(1, n_pages + 1), n_try)


def add_users_from_anime(media, known, new_rows, limit):
    """
    Append up to `limit` users who have this seed anime on their list (and scored it, if
    SEED_REQUIRE_SCORED) and aren't listed yet to new_rows. Returns how many were added.
    Rows are appended as they're found, so they survive an error part-way through the anime.
    """
    taken = 0
    for page in candidate_pages(media):
        data = post_graphql(USERS_QUERY, {"mediaId": media["id"], "page": page, "perPage": PER_PAGE})
        for entry in data["Page"]["mediaList"]:
            user = entry["user"]
            # Entries from deleted accounts come back with user: null
            if user is None or (SEED_REQUIRE_SCORED and not entry["score"]) or user["name"].lower() in known:
                continue
            known.add(user["name"].lower())
            new_rows.append({"username": user["name"], "username_key": user["name"].lower(),
                             "anilist_user_id": user["id"], "source": "seed", "status": "pending",
                             "found_via_anime_id": media["id"], "added_at": now_iso()})
            taken += 1
            if taken >= limit:
                return taken
    return taken


def collect_seed_usernames(seed_state, usernames):
    """Up to USERS_PER_ANIME new users per seed anime, until MAX_USERS seed users are listed."""
    seed_count = int((usernames["source"] == "seed").sum())
    todo = [a for a in seed_state["anime"][:N_SEED_ANIME] if not a["users_collected"]]
    if seed_count >= MAX_USERS or not todo:
        print(f"User discovery: nothing to do ({seed_count} seed users listed, {len(todo)} seed anime left), no requests needed")
        return usernames

    known = set(usernames["username_key"])
    new_rows = []
    start, stopped = REQUEST_STATS["requests"], None

    def flush():
        nonlocal usernames
        if new_rows:
            usernames = pd.concat([usernames, pd.DataFrame(new_rows)], ignore_index=True)
            new_rows.clear()
        save_usernames(usernames)
        write_json(SEED_PATH, seed_state)

    progress = tqdm(todo, desc="Users from seed anime")
    try:
        for media in progress:
            if seed_count >= MAX_USERS:
                stopped = f"MAX_USERS ({MAX_USERS}) reached"
                break
            if out_of_time():
                stopped = "MAX_RUNTIME_MINUTES reached"
                break
            rows_before = len(new_rows)
            try:
                taken = add_users_from_anime(media, known, new_rows, limit=min(USERS_PER_ANIME, MAX_USERS - seed_count))
            except (AniListError, RetriesExhausted) as exc:
                # Left as not collected, so the next run tries this anime again
                seed_count += len(new_rows) - rows_before
                log(f"seed anime {media['id']}: {exc}")
                REQUEST_STATS["seed_anime_failed"] += 1
                continue
            seed_count += taken
            media["users_collected"] = True
            media["users_found"] = taken
            flush()
            progress.set_postfix(seed_users=seed_count)
    except KeyboardInterrupt:
        flush()
        print(f"Interrupted. Saved {seed_count} seed users; the next run continues from the next seed anime.")
        raise
    except Exception:
        flush()  # keep what was collected before the error
        raise
    flush()

    n_done = sum(a["users_collected"] for a in seed_state["anime"][:N_SEED_ANIME])
    print(f"User discovery: {n_done} of {min(N_SEED_ANIME, len(seed_state['anime']))} seed anime searched, "
          f"{seed_count} seed users ({REQUEST_STATS['requests'] - start} requests)"
          + (f". Stopped: {stopped}" if stopped else ""))
    if REQUEST_STATS["seed_anime_failed"]:
        print(f"  {REQUEST_STATS['seed_anime_failed']} seed anime failed and will be retried next run (see {LOG_PATH.as_posix()})")
    return usernames


usernames = register_manual_users(load_usernames())
if ONLY_EXTRA_USERS:
    print("ONLY_EXTRA_USERS: skipping user discovery")
else:
    usernames = collect_seed_usernames(seed_state, usernames)
print(f"Unique users listed: {usernames['username_key'].nunique()}")

Manual users: 1 requested, 0 newly added as pending, 1 already listed
User discovery: nothing to do (10000 seed users listed, 218 seed anime left), no requests needed
Unique users listed: 10001


In [10]:
# Validation: usernames
check = Check()
usernames = load_usernames()
requested_manual = [n.lower() for n in read_extra_usernames()]
seed_users = usernames[usernames["source"] == "seed"]

check(not usernames["username_key"].duplicated().any(), f"{len(usernames)} usernames, no case-insensitive duplicates")
check(usernames["status"].isin(USER_STATUSES).all(), "all statuses are valid")
check(usernames["source"].isin(["seed", "manual"]).all(), "all sources are seed or manual")
if len(seed_users) > MAX_USERS:
    # MAX_USERS only caps discovery: users collected under a higher cap are kept
    print(f"[NOTE] {len(seed_users)} seed users already listed, more than MAX_USERS ({MAX_USERS}). "
          "No more will be discovered; lowering MAX_USERS doesn't remove users already collected.")
else:
    check(True, f"{len(seed_users)} seed users, within MAX_USERS ({MAX_USERS})")
check(seed_users["anilist_user_id"].notna().all(), "every seed user has an AniList user ID")
check(set(requested_manual) <= set(usernames["username_key"]), f"all {len(requested_manual)} requested manual users are listed")
display(pd.crosstab(usernames["source"], usernames["status"], margins=True))
check.done()

[PASS] 10001 usernames, no case-insensitive duplicates
[PASS] all statuses are valid
[PASS] all sources are seed or manual
[NOTE] 10000 seed users already listed, more than MAX_USERS (1000). No more will be discovered; lowering MAX_USERS doesn't remove users already collected.
[PASS] every seed user has an AniList user ID
[PASS] all 1 requested manual users are listed


status,done,private,All
source,,,
manual,1,0,1
seed,9482,518,10000
All,9483,518,10001


## 3–4. User lists

In [11]:
def fuzzy_date(d):
    """AniList FuzzyDate as 'YYYY-MM-DD', 'YYYY-MM' or 'YYYY' (None without a year)."""
    if not d or not d.get("year"):
        return None
    text = f"{d['year']:04d}"
    if d.get("month"):
        text += f"-{d['month']:02d}"
        if d.get("day"):
            text += f"-{d['day']:02d}"
    return text


def unix_to_iso(ts):
    return datetime.fromtimestamp(ts, timezone.utc).isoformat(timespec="seconds") if ts else None


def fetch_user_list(row):
    """A user's MediaListCollection, by AniList ID when known (survives renames), otherwise by name."""
    if pd.notna(row.anilist_user_id):
        query, variables = LIST_BY_ID_QUERY, {"userId": int(row.anilist_user_id)}
    else:
        query, variables = LIST_BY_NAME_QUERY, {"userName": row.username}
    collection = post_graphql(query, {**variables, "scoreFormat": SCORE_FORMAT})["MediaListCollection"]

    if collection["hasNextChunk"]:
        # Very large lists come back in chunks: refetch the whole list chunk by chunk
        lists, chunk = [], 1
        while True:
            part = post_graphql(LIST_CHUNK_QUERY, {"userId": collection["user"]["id"], "scoreFormat": SCORE_FORMAT,
                                                   "chunk": chunk, "perChunk": 500})["MediaListCollection"]
            lists += part["lists"]
            if not part["hasNextChunk"]:
                break
            chunk += 1
        collection = {**collection, "lists": lists}
    return collection


def list_rows(user_id, lists):
    """One row per entry. Custom lists repeat entries from the status lists, so they are skipped."""
    return [
        {
            "anilist_user_id": user_id,
            "anime_id": e["mediaId"],
            "status": e["status"],
            "score": e["score"],
            "progress": e["progress"],
            "repeat": e["repeat"],
            "started_at": fuzzy_date(e["startedAt"]),
            "completed_at": fuzzy_date(e["completedAt"]),
            "updated_at": unix_to_iso(e["updatedAt"]),
            "created_at": unix_to_iso(e["createdAt"]),
        }
        for lst in lists if not lst["isCustomList"]
        for e in lst["entries"]
    ]


def fetch_user(row):
    """Fetch one user's list. Returns the update for their usernames.csv row and their entries."""
    update = {"fetched_at": now_iso(), "error": ""}
    try:
        collection = fetch_user_list(row)
    except PrivateUser:
        return {**update, "status": "private"}, []
    except UserNotFound:
        return {**update, "status": "not_found"}, []
    except (AniListError, RetriesExhausted) as exc:
        return {**update, "status": "failed", "error": str(exc)[:300]}, []

    user = collection["user"]
    entries = list_rows(user["id"], collection["lists"])
    entries = drop_duplicate_entries(pd.DataFrame(entries, columns=PART_COLUMNS)).to_dict("records")
    stats = (user.get("statistics") or {}).get("anime") or {}
    return {
        **update,
        "status": "done",
        "anilist_user_id": user["id"],
        "username": user["name"],
        "username_key": user["name"].lower(),
        "n_entries": len(entries),
        "episodes_watched": sum(e["progress"] or 0 for e in entries),
        # AniList's statistics can be empty for older accounts; leave watch time unknown then
        "minutes_watched": stats.get("minutesWatched") if stats.get("count") else pd.NA,
    }, entries


def fetch_user_lists(usernames):
    """Fetch every pending user's list (manual users first), checkpointing every CHECKPOINT_EVERY users."""
    statuses = ["pending"] + (["failed"] if RETRY_FAILED else [])
    todo = usernames[usernames["status"].isin(statuses)]
    if ONLY_EXTRA_USERS:
        todo = todo[todo["source"] == "manual"]
    todo = pd.concat([todo[todo["source"] == "manual"], todo[todo["source"] != "manual"]])
    n_waiting = len(todo)
    if MAX_USERS_THIS_RUN is not None:
        todo = todo.head(MAX_USERS_THIS_RUN)
    if todo.empty:
        print("User lists: nothing pending, no requests needed")
        return usernames

    rows, updates = [], {}
    start, stopped, n_rows_written, n_done = REQUEST_STATS["requests"], None, 0, 0

    def flush():
        # Entries first, then statuses: if interrupted in between, those users are fetched again
        # and the duplicate fetch is dropped at assembly
        nonlocal n_rows_written
        if rows:
            write_csv(pd.DataFrame(rows, columns=PART_COLUMNS), next_part_path(RATINGS_PARTS_DIR, ".csv.gz"))
            n_rows_written += len(rows)
        for index, values in updates.items():
            for column, value in values.items():
                usernames.at[index, column] = value
        save_usernames(usernames)
        rows.clear()
        updates.clear()

    progress = tqdm(list(todo.itertuples()), desc="User lists")
    try:
        for row in progress:
            if out_of_time():
                stopped = "MAX_RUNTIME_MINUTES reached"
                break
            update, entries = fetch_user(row)
            rows.extend(entries)
            updates[row.Index] = update
            if row.source == "manual" and update["status"] != "done":
                print(f"  manual user '{row.username}': {update['status']}")
            log(f"user {row.username}: {update['status']} {update['error']}".rstrip())
            n_done += 1
            progress.set_postfix(entries=n_rows_written + len(rows))
            if len(updates) >= CHECKPOINT_EVERY:
                flush()
        else:
            if MAX_USERS_THIS_RUN is not None and n_waiting > MAX_USERS_THIS_RUN:
                stopped = f"MAX_USERS_THIS_RUN ({MAX_USERS_THIS_RUN}) reached"
    except KeyboardInterrupt:
        flush()
        print(f"Interrupted after {n_done} users; progress saved. Rerun to continue with the remaining users.")
        raise
    except Exception:
        flush()  # keep the users fetched before the error
        raise
    flush()

    status = usernames["status"].value_counts()
    print(f"User lists: {n_done} users fetched this run ({REQUEST_STATS['requests'] - start} requests), "
          f"{n_rows_written} entries written" + (f". Stopped: {stopped}" if stopped else ""))
    print("  " + ", ".join(f"{s}: {int(status.get(s, 0))}" for s in ["done", "pending", "private", "not_found", "failed"]))
    return usernames


usernames = fetch_user_lists(load_usernames())

User lists: nothing pending, no requests needed


In [12]:
# Validation: user lists (checkpoint parts, latest fetch per user)
check = Check()
usernames = load_usernames()
parts = sorted(RATINGS_PARTS_DIR.glob("part_*.csv.gz"))
raw_entries = pd.concat([pd.read_csv(p) for p in parts], ignore_index=True) if parts else pd.DataFrame(columns=PART_COLUMNS)
latest_entries = pd.concat(list(iter_latest_entries(dedupe=False)), ignore_index=True) if parts else raw_entries
entries = drop_duplicate_entries(latest_entries)

max_score = SCORE_MAX[SCORE_FORMAT]
done = usernames[usernames["status"] == "done"]
expected_rows = int(done["n_entries"].sum())
print(f"{len(parts)} part files, {len(raw_entries)} rows written, {len(latest_entries)} in users' latest fetches, "
      f"{len(latest_entries) - len(entries)} duplicate list entries dropped (AniList can list an anime twice)")
check(not entries.duplicated(["anilist_user_id", "anime_id"]).any(), "no duplicate user/anime pairs (latest fetch per user)")
check(entries["anilist_user_id"].notna().all() and entries["anime_id"].notna().all(), "no missing user or anime IDs")
check(entries["score"].between(0, max_score).all(), f"scores within 0-{max_score} ({SCORE_FORMAT})")
if SCORE_FORMAT != "POINT_10_DECIMAL":
    check((entries["score"] % 1 == 0).all(), "scores are whole numbers")
# Users fetched before duplicate entries were dropped at fetch time have them in n_entries; accept either count
n_expected = done.loc[done["n_entries"] > 0].set_index("anilist_user_id")["n_entries"]
n_expected.index = n_expected.index.astype("int64")
n_raw = latest_entries.groupby("anilist_user_id").size().reindex(n_expected.index)
n_kept = entries.groupby("anilist_user_id").size().reindex(n_expected.index)
check(((n_expected == n_raw) | (n_expected == n_kept)).all(), f"rows match the entry counts of done users ({expected_rows})")
check(set(entries["anilist_user_id"]) == set(done.loc[done["n_entries"] > 0, "anilist_user_id"]), "every done user with entries has rows, and no others")
manual = usernames[usernames["source"] == "manual"]
check(set(manual.loc[manual["status"] == "done", "anilist_user_id"]) <= set(entries["anilist_user_id"]) | set(done.loc[done["n_entries"] == 0, "anilist_user_id"]),
      f"manual users fetched: {manual['status'].value_counts().to_dict()}")
if len(entries):
    print(f"Rated: {(entries['score'] > 0).sum()} of {len(entries)} entries; statuses: {entries['status'].value_counts().to_dict()}")
    display(hide_usernames(entries).head())
check.done()

103 part files, 7562533 rows written, 7485282 in users' latest fetches, 69 duplicate list entries dropped (AniList can list an anime twice)
[PASS] no duplicate user/anime pairs (latest fetch per user)
[PASS] no missing user or anime IDs
[PASS] scores within 0-100 (POINT_100)
[PASS] scores are whole numbers
[PASS] rows match the entry counts of done users (7485282)
[PASS] every done user with entries has rows, and no others
[PASS] manual users fetched: {'done': 1}
Rated: 4557327 of 7485213 entries; statuses: {'COMPLETED': 4751662, 'PLANNING': 1972678, 'DROPPED': 350754, 'CURRENT': 225049, 'PAUSED': 178190, 'REPEATING': 6880}


,anime_id,status,score,progress,repeat,started_at,completed_at,updated_at,created_at
0,6682,COMPLETED,60,12,0,NaN,NaN,NaN,NaN
1,21366,COMPLETED,100,22,0,NaN,NaN,NaN,NaN
2,98478,COMPLETED,100,22,0,NaN,NaN,NaN,NaN
3,100526,COMPLETED,70,12,0,NaN,2018-06-21,2019-03-15T12:34:08+00:00,NaN
4,99725,COMPLETED,80,12,0,NaN,NaN,NaN,NaN


## 5. Anime metadata

In [13]:
def fetch_metadata(seed_state, usernames):
    """Fetch metadata for the target anime not fetched yet, 50 per request."""
    if not users_complete(usernames) and not ASSEMBLE_PARTIAL:
        status = usernames["status"].value_counts()
        print(f"Anime metadata: skipped until every user is done ({int(status.get('pending', 0))} pending, "
              f"{int(status.get('failed', 0))} failed). Set RETRY_FAILED or ASSEMBLE_PARTIAL to proceed.")
        return
    targets, _ = metadata_targets(seed_state)
    progress = load_metadata_progress()
    finished = set(progress["done"]) | set(progress["missing"])
    pending = [i for i in targets if i not in finished]
    if not pending:
        print(f"Anime metadata: all {len(targets)} target anime fetched, no requests needed")
        return

    start, stopped = REQUEST_STATS["requests"], None
    batches = [pending[i:i + PER_PAGE] for i in range(0, len(pending), PER_PAGE)]
    try:
        for batch in tqdm(batches, desc="Metadata batches"):
            if out_of_time():
                stopped = "MAX_RUNTIME_MINUTES reached"
                break
            try:
                media = post_graphql(METADATA_QUERY, {"ids": batch, "perPage": PER_PAGE})["Page"]["media"]
            except (AniListError, RetriesExhausted) as exc:
                log(f"metadata batch starting {batch[0]}: {exc}")  # retried next run
                continue
            # Part first, then the progress file: an interrupt in between only causes a refetch
            atomic_write(next_part_path(METADATA_PARTS_DIR, ".jsonl"),
                         lambda tmp: tmp.write_text("".join(json.dumps(m) + "\n" for m in media), encoding="utf-8"))
            returned = {m["id"] for m in media}
            progress["done"] += sorted(returned)
            progress["missing"] += [i for i in batch if i not in returned]
            progress["updated_at"] = now_iso()
            write_json(METADATA_DONE_PATH, progress)
    except KeyboardInterrupt:
        print(f"Interrupted; {len(progress['done'])} anime with metadata saved. Rerun to continue.")
        raise
    print(f"Anime metadata: {len(progress['done'])} fetched, {len(progress['missing'])} not returned by AniList "
          f"({REQUEST_STATS['requests'] - start} requests)" + (f". Stopped: {stopped}" if stopped else ""))


fetch_metadata(seed_state, load_usernames())

Anime metadata: all 10500 target anime fetched, no requests needed


In [14]:
def clean_description(text):
    """AniList descriptions contain <br>, <i> and HTML entities even with asHtml: false."""
    if not text:
        return None
    text = re.sub(r"<br\s*/?>", "\n", text, flags=re.IGNORECASE)
    text = html.unescape(re.sub(r"<[^>]+>", "", text))
    # Cut the "(Source: ...)" credit and anything after it (broadcast notes, not plot)
    text = re.split(r"\(Sources?\s*:", text, maxsplit=1, flags=re.IGNORECASE)[0]
    return re.sub(r"\n{3,}", "\n\n", text).strip() or None


def all_time_rank(media, kind):
    return next((r["rank"] for r in media.get("rankings") or [] if r["type"] == kind and r["allTime"]), None)


def day_month_year(d):
    """'dd/mm/yyyy', only for complete dates."""
    if not (d and d.get("year") and d.get("month") and d.get("day")):
        return None
    return f"{d['day']:02d}/{d['month']:02d}/{d['year']}"


def flatten_media(m, seed_ranks, counts, mal_synopses):
    """One anime_metadata.csv row from an AniList Media record."""
    tags = sorted(m["tags"] or [], key=lambda t: -t["rank"])
    themes = [t["name"] for t in tags if t["rank"] >= THEME_MIN_RANK and not t["isMediaSpoiler"]]
    anilist_synopsis = clean_description(m["description"])
    mal_synopsis = mal_synopses.get(m["idMal"]) or None
    score_distribution = (m["stats"] or {}).get("scoreDistribution") or []
    return {
        "anime_ID": m["id"],
        "Title": m["title"]["romaji"],
        "Type": m["format"].lower() if m["format"] else None,
        "Mean Rating": m["meanScore"] / 10 if m["meanScore"] else None,
        "Rank": all_time_rank(m, "RATED"),
        "Popularity": all_time_rank(m, "POPULAR"),
        "Year": m["seasonYear"] or (m["startDate"] or {}).get("year"),
        "Season": f"{m['season'].lower()} {m['seasonYear']}" if m["season"] and m["seasonYear"] else None,
        "Genre(s)": ", ".join(m["genres"] or []) or None,
        "Theme(s)": ", ".join(themes) or None,
        "Episode Count": m["episodes"],
        "Synopsis": mal_synopsis or anilist_synopsis,  # MAL's synopses are usually fuller
        "Start Date": day_month_year(m["startDate"]),
        "End Date": day_month_year(m["endDate"]),
        "mal_ID": m["idMal"],
        "synopsis_source": "mal" if mal_synopsis else ("anilist" if anilist_synopsis else None),
        "synopsis_anilist": anilist_synopsis,
        "title_english": m["title"]["english"],
        "title_native": m["title"]["native"],
        "duration_min": m["duration"],
        "airing_status": m["status"],
        "source": m["source"],
        "studios": ", ".join(n["name"] for n in m["studios"]["nodes"]) or None,
        "tags": "; ".join(f"{t['name']}:{t['rank']}" for t in tags) or None,
        "average_score": m["averageScore"],
        "mean_score": m["meanScore"],
        "anilist_popularity": m["popularity"],
        "favourites": m["favourites"],
        "is_adult": m["isAdult"],
        "score_distribution": json.dumps({d["score"]: d["amount"] for d in score_distribution}),
        "seed_rank": seed_ranks.get(m["id"]),
        "n_ratings_in_sample": int(counts.get(m["id"], 0)),
    }


def load_metadata(seed_state):
    """anime_metadata.csv as a DataFrame, built from the checkpoints, most-rated anime first."""
    seed_ranks = {a["id"]: a["seed_rank"] for a in (seed_state or {}).get("anime", [])}
    counts = rated_counts()
    found = load_mal_synopses().query("status == 'found'")
    mal_synopses = dict(zip(found["mal_id"], found["synopsis"]))
    df = pd.DataFrame([flatten_media(m, seed_ranks, counts, mal_synopses) for m in metadata_records().values()])
    if not df.empty:
        int_columns = ["Rank", "Popularity", "Year", "Episode Count", "mal_ID", "duration_min", "average_score",
                       "mean_score", "anilist_popularity", "favourites", "seed_rank"]
        df[int_columns] = df[int_columns].astype("Int64")
        df = df.sort_values(["n_ratings_in_sample", "anime_ID"], ascending=[False, True]).reset_index(drop=True)
    return df


# Validation: anime metadata
check = Check()
metadata = load_metadata(seed_state)
meta_progress = load_metadata_progress()
if metadata.empty:
    print("No metadata fetched yet")
else:
    targets, _ = metadata_targets(seed_state)
    check(not metadata["anime_ID"].duplicated().any(), f"{len(metadata)} anime, no duplicate IDs")
    check(metadata["anime_ID"].notna().all() and metadata["Title"].notna().all(), "no missing IDs or titles")
    check(set(targets) <= metadata_finished(), f"all {len(targets)} target anime fetched or confirmed missing ({len(meta_progress['missing'])} missing)")
    if seed_state:
        top_rated = {a["id"] for a in seed_state["anime"][:N_TOP_RATED]}
        check(top_rated <= set(metadata["anime_ID"]) | set(meta_progress["missing"]), f"top {len(top_rated)} seed anime included")
    for column in ["mal_ID", "Synopsis", "Genre(s)", "Theme(s)", "Episode Count", "Rank"]:
        print(f"  {column:<14}{metadata[column].notna().mean():6.1%} filled")
    print(f"  Synopsis from: {metadata['synopsis_source'].value_counts().to_dict()} (MAL synopses are fetched in the next stage)")
    display(metadata[["anime_ID", "Title", "Type", "Mean Rating", "Year", "Season", "Genre(s)", "Theme(s)", "Episode Count", "mal_ID", "n_ratings_in_sample"]].head())
check.done()

[PASS] 15000 anime, no duplicate IDs
[PASS] no missing IDs or titles
[PASS] all 10500 target anime fetched or confirmed missing (0 missing)
[PASS] top 100 seed anime included
  mal_ID         97.8% filled
  Synopsis       99.6% filled
  Genre(s)       93.7% filled
  Theme(s)       82.5% filled
  Episode Count  99.3% filled
  Rank            6.7% filled
  Synopsis from: {'mal': 14353, 'anilist': 587} (MAL synopses are fetched in the next stage)


,anime_ID,Title,Type,Mean Rating,Year,Season,Genre(s),Theme(s),Episode Count,mal_ID,n_ratings_in_sample
0,16498,Shingeki no Kyojin,tv,8.5,2013,spring 2013,"Action, Drama, Fantasy, Mystery","Kaiju, Revenge, Military, Primarily Teen Cast,...",25,16498,7391
1,1535,DEATH NOTE,tv,8.4,2006,fall 2006,"Mystery, Psychological, Supernatural, Thriller","Crime, Detective, Anti-Hero, Male Protagonist,...",37,1535,6631
2,11757,Sword Art Online,tv,7.0,2012,summer 2012,"Action, Adventure, Fantasy, Romance","Virtual World, Video Games, Male Protagonist, ...",25,11757,6406
3,21087,One Punch Man,tv,8.3,2015,fall 2015,"Action, Comedy, Sci-Fi, Supernatural","Superhero, Super Power, Parody, Male Protagoni...",12,30276,6135
4,21459,Boku no Hero Academia,tv,7.7,2016,spring 2016,"Action, Adventure, Comedy","Super Power, Superhero, Shounen, Primarily Tee...",13,31964,6058


## 5b. MAL synopses

AniList's descriptions are often short. This looks up each anime's synopsis on MyAnimeList by its MAL ID, and assembly uses it in place of AniList's where there is one.

- **Source:** the official MAL API when the `MAL_CLIENT_ID` environment variable is set (a free client ID from [myanimelist.net/apiconfig](https://myanimelist.net/apiconfig)), otherwise [Jikan](https://jikan.moe), an unofficial MAL API that needs no key. Set it before starting Jupyter, e.g. `set MAL_CLIENT_ID=...` on Windows. It is read from the environment so it never ends up in the notebook.
- **Speed:** one request per anime at `MAL_REQUESTS_PER_MINUTE`, checkpointed every `CHECKPOINT_EVERY` lookups. It resumes like the other stages and respects `MAX_RUNTIME_MINUTES`.
- **Unreachable API:** if the first lookups all fail (for example, a VPN blocking Jikan), the stage stops with a message instead of retrying every anime.
- **Licensing:** synopses are MAL's copyrighted text. They stay in the gitignored output, and `scripts/anonymise_users.py` keeps them out of the committed data. Only the embeddings built from them are committed.

In [15]:
JIKAN_URL = "https://api.jikan.moe/v4/anime/{}"
MAL_API_URL = "https://api.myanimelist.net/v2/anime/{}?fields=synopsis"
MAL_THROTTLE = Throttle()


class MalNotFound(Exception):
    """The MAL ID doesn't exist on MAL."""


class MalConfigError(Exception):
    """The MAL API rejected the client ID. Not retried."""


def mal_source():
    if MAL_SOURCE == "auto":
        return "mal_api" if MAL_CLIENT_ID else "jikan"
    return MAL_SOURCE


def mal_request(mal_id):
    """URL and headers to look up one anime on the configured MAL source."""
    if mal_source() == "mal_api":
        return MAL_API_URL.format(mal_id), {"X-MAL-CLIENT-ID": MAL_CLIENT_ID}
    return JIKAN_URL.format(mal_id), {}


def clean_mal_synopsis(text):
    """Drop MAL's credit lines and its 'no synopsis' placeholder."""
    text = (text or "").strip()
    text = re.sub(r"\s*\[Written by MAL Rewrite\]\s*$", "", text)
    text = re.split(r"\(Sources?\s*:", text, maxsplit=1, flags=re.IGNORECASE)[0]  # credit and any notes after it
    if text.lower().startswith("no synopsis"):
        return ""
    return text.strip()


def get_mal_synopsis(mal_id):
    """
    One anime's MAL synopsis ('' if MAL has none), throttled to MAL_REQUESTS_PER_MINUTE.

    404 raises MalNotFound. 429 (and 403 from the MAL API, which it uses when throttling) waits and
    retries. 5xx and network errors back off exponentially, up to MAX_RETRIES, then raise RetriesExhausted.
    """
    source = mal_source()
    if source == "mal_api" and not MAL_CLIENT_ID:
        raise MalConfigError("MAL_SOURCE is 'mal_api' but the MAL_CLIENT_ID environment variable isn't set")
    url, headers = mal_request(mal_id)

    failures = 0
    while True:
        MAL_THROTTLE.wait(MAL_REQUESTS_PER_MINUTE)
        REQUEST_STATS["mal_requests"] += 1
        try:
            response = requests.get(url, headers=headers, timeout=30)
        except requests.RequestException as exc:
            problem = f"{type(exc).__name__}: {exc}"
        else:
            status = response.status_code
            if status == 200:
                payload = response.json()
                return clean_mal_synopsis(payload["data"].get("synopsis") if source == "jikan" else payload.get("synopsis"))
            if status == 404:
                raise MalNotFound(mal_id)
            if status in (400, 401):
                raise MalConfigError(f"HTTP {status} from the MAL API: check MAL_CLIENT_ID")
            if status == 429 or (status == 403 and source == "mal_api"):
                wait = retry_after_seconds(response, default=30)
                REQUEST_STATS["mal_rate_limited"] += 1
                log(f"MAL HTTP {status}, waiting {wait}s")
                time.sleep(wait + 1)
                continue
            problem = f"HTTP {status}"

        failures += 1
        back_off(failures, problem, max_delay=60, label=f"MAL {mal_id}: ")


def mal_reachable(mal_id):
    """One quick request to check the API can be reached at all; any HTTP response counts."""
    source = mal_source()
    if source == "mal_api" and not MAL_CLIENT_ID:
        print("MAL synopses: MAL_SOURCE is 'mal_api' but the MAL_CLIENT_ID environment variable isn't set")
        return False
    url, headers = mal_request(mal_id)
    REQUEST_STATS["mal_requests"] += 1
    try:
        requests.get(url, headers=headers, timeout=15)
        return True
    except requests.RequestException as exc:
        hint = ("A VPN or firewall may be blocking it: try without one, or set MAL_CLIENT_ID to use the official MAL API."
                if source == "jikan" else "Check your connection.")
        print(f"MAL synopses: can't reach {source} ({type(exc).__name__}), so nothing was fetched. {hint}")
        return False


def look_up_synopsis(mal_id):
    """A checkpoint row for one MAL ID. RetriesExhausted is passed on, so the ID isn't recorded and is retried next run."""
    try:
        text = get_mal_synopsis(mal_id)
        status = "found" if text else "empty"
    except MalNotFound:
        text, status = "", "not_found"
    return {"mal_id": mal_id, "synopsis": text, "status": status, "fetched_at": now_iso()}


def fetch_mal_synopses():
    """Look up the MAL synopsis of every anime with metadata and a MAL ID, checkpointing every CHECKPOINT_EVERY lookups."""
    if not FETCH_MAL_SYNOPSES:
        print("MAL synopses: FETCH_MAL_SYNOPSES is False, AniList descriptions will be used")
        return
    n_mal_ids = len(mal_ids_with_metadata())
    pending = sorted(mal_synopses_left())
    if not pending:
        print(f"MAL synopses: all {n_mal_ids} looked up, no requests needed")
        return

    source = mal_source()
    if not mal_reachable(pending[0]):
        return
    print(f"MAL synopses: {len(pending)} of {n_mal_ids} to look up via {source} "
          f"(~{len(pending) / MAL_REQUESTS_PER_MINUTE:.0f} min at {MAL_REQUESTS_PER_MINUTE}/min)")
    rows, stopped = [], None
    start = REQUEST_STATS["mal_requests"]
    n_failed = consecutive_failures = 0

    def flush():
        if rows:
            write_csv(pd.DataFrame(rows, columns=MAL_SYNOPSIS_COLUMNS), next_part_path(MAL_SYNOPSIS_PARTS_DIR, ".csv"))
            rows.clear()

    try:
        for mal_id in tqdm(pending, desc="MAL synopses"):
            if out_of_time():
                stopped = "MAX_RUNTIME_MINUTES reached"
                break
            try:
                rows.append(look_up_synopsis(mal_id))
            except RetriesExhausted as exc:
                log(f"MAL {mal_id}: {exc}")
                n_failed += 1
                consecutive_failures += 1
                if consecutive_failures >= 5:
                    stopped = f"{source} failed 5 lookups in a row ({exc}); it may be down, so try again later"
                    break
                continue
            consecutive_failures = 0
            if len(rows) >= CHECKPOINT_EVERY:
                flush()
    except KeyboardInterrupt:
        flush()
        print("Interrupted; lookups so far are saved. Rerun to continue.")
        raise
    except Exception:
        flush()  # keep the lookups made before the error
        raise
    flush()

    print(f"MAL synopses: {len(load_mal_synopses())} of {n_mal_ids} looked up ({REQUEST_STATS['mal_requests'] - start} requests"
          + (f", {n_failed} failed and will be retried next run" if n_failed else "") + ")"
          + (f". Stopped: {stopped}" if stopped else ""))


fetch_mal_synopses()

MAL synopses: all 14660 looked up, no requests needed


In [16]:
# Validation: MAL synopses
check = Check()
found = load_mal_synopses()
if not FETCH_MAL_SYNOPSES or found.empty:
    print("No MAL synopses looked up")
else:
    mal_ids = mal_ids_with_metadata()
    check(not found["mal_id"].duplicated().any(), f"{len(found)} lookups, no duplicate MAL IDs")
    check(found["status"].isin(["found", "empty", "not_found"]).all(), "all statuses are valid")
    check(set(found["mal_id"]) <= mal_ids, "every lookup is for an anime with metadata")
    check((found.loc[found["status"] == "found", "synopsis"].str.len() > 0).all(), "every 'found' synopsis has text")
    print(f"Looked up {len(set(found['mal_id']) & mal_ids)} of {len(mal_ids)} MAL IDs: {found['status'].value_counts().to_dict()}")

    anilist_words = pd.Series([len((clean_description(m["description"]) or "").split()) for m in metadata_records().values() if m["idMal"]])
    mal_words = found.loc[found["status"] == "found", "synopsis"].str.split().str.len()
    print(f"Median synopsis length: MAL {mal_words.median():.0f} words vs AniList {anilist_words.median():.0f} words")
check.done()

[PASS] 14660 lookups, no duplicate MAL IDs
[PASS] all statuses are valid
[PASS] every lookup is for an anime with metadata
[PASS] every 'found' synopsis has text
Looked up 14660 of 14660 MAL IDs: {'found': 14339, 'empty': 290, 'not_found': 31}
Median synopsis length: MAL 74 words vs AniList 61 words


## 6. Assembly

In [17]:
RATING_COLUMNS = ["anilist_user_id", "username", "source", "anime_id", "anime_title", "status", "score", "rating",
                  "is_rated", "progress", "repeat", "started_at", "completed_at", "updated_at", "created_at"]
PROFILE_STATUSES = {"Watching": ["CURRENT", "REPEATING"], "Completed": ["COMPLETED"], "On-Hold": ["PAUSED"],
                    "Dropped": ["DROPPED"], "Plan to Watch": ["PLANNING"]}
PROFILE_COUNT_COLUMNS = ["Total Entries", "Rewatched", "Episodes", *PROFILE_STATUSES]
PROFILE_COLUMNS = ["user", "Gender", "Watching", "Completed", "On-Hold", "Dropped", "Plan to Watch",
                   "Total Entries", "Rewatched", "Episodes", "Days", "Mean Score", "anilist_user_id", "source"]


def ready_to_assemble(seed_state, usernames):
    """Whether every stage is complete; prints what's missing if not. ASSEMBLE_PARTIAL skips the checks."""
    if ASSEMBLE_PARTIAL:
        return True
    if not users_complete(usernames):
        print("Assembly: skipped until every user is done. Set ASSEMBLE_PARTIAL = True to assemble what has been collected.")
        return False
    targets, _ = metadata_targets(seed_state)
    if not set(targets) <= metadata_finished():
        print("Assembly: skipped until anime metadata is complete. Set ASSEMBLE_PARTIAL = True to assemble anyway.")
        return False
    n_left = len(mal_synopses_left()) if FETCH_MAL_SYNOPSES else 0
    if n_left:
        print(f"Assembly: skipped until MAL synopses are complete ({n_left} left). "
              "Set ASSEMBLE_PARTIAL = True to assemble anyway, or FETCH_MAL_SYNOPSES = False to use AniList's.")
        return False
    return True


def profile_counts(entries):
    """Per-user list statistics for one chunk of entries (every entry, rated or not)."""
    by_user = entries.groupby("anilist_user_id")
    profile = pd.DataFrame({
        "Total Entries": by_user.size(),
        "Rewatched": by_user["repeat"].sum(),
        "Episodes": by_user["progress"].sum(),
        "Mean Score": by_user["rating"].mean().round(2),
    })
    for column, statuses in PROFILE_STATUSES.items():
        profile[column] = entries["status"].isin(statuses).groupby(entries["anilist_user_id"]).sum()
    return profile


def assemble(seed_state, usernames):
    """Combine the checkpoints into ratings.csv.gz, anime_metadata.csv, user_profiles.csv and users.csv."""
    if not ready_to_assemble(seed_state, usernames):
        return False

    metadata = load_metadata(seed_state)
    titles = metadata.set_index("anime_ID")["Title"] if not metadata.empty else pd.Series(dtype=object)
    done = usernames[usernames["status"] == "done"].copy()
    done["anilist_user_id"] = done["anilist_user_id"].astype("int64")  # done users always have an ID
    lookup = done.set_index("anilist_user_id")
    scale = 10 / SCORE_MAX[SCORE_FORMAT]
    profile_parts = []
    n_rows = 0

    def write_ratings(tmp):
        nonlocal n_rows
        with gzip.open(tmp, "wt", encoding="utf-8", newline="") as f:
            first = True
            for df in iter_latest_entries():
                df = df.join(lookup[["username", "source"]], on="anilist_user_id")
                df["anime_title"] = df["anime_id"].map(titles)
                df["is_rated"] = df["score"] > 0
                df["rating"] = (df["score"] * scale).where(df["is_rated"])
                profile_parts.append(profile_counts(df))  # before unrated entries are dropped

                if DROP_UNRATED:
                    df = df[df["is_rated"]]
                df[RATING_COLUMNS].to_csv(f, index=False, header=first)
                first = False
                n_rows += len(df)
            if first:
                f.write(",".join(RATING_COLUMNS) + "\n")

    atomic_write(OUTPUT_DIR / "ratings.csv.gz", write_ratings)

    # One profile per done user, including users with empty lists
    profiles = pd.concat(profile_parts) if profile_parts else pd.DataFrame(columns=[*PROFILE_COUNT_COLUMNS, "Mean Score"])
    profiles = profiles.reindex(pd.Index(done["anilist_user_id"], name="anilist_user_id")).reset_index()
    profiles[PROFILE_COUNT_COLUMNS] = profiles[PROFILE_COUNT_COLUMNS].fillna(0).astype("int64")
    profiles["user"] = profiles["anilist_user_id"].map(lookup["username"])
    profiles["source"] = profiles["anilist_user_id"].map(lookup["source"])
    profiles["Gender"] = pd.NA  # not available from AniList
    profiles["Days"] = (profiles["anilist_user_id"].map(lookup["minutes_watched"]).astype("Float64") / 1440).round(1)

    write_csv(profiles[PROFILE_COLUMNS], OUTPUT_DIR / "user_profiles.csv")
    write_csv(metadata, OUTPUT_DIR / "anime_metadata.csv")
    write_csv(usernames[["username", "anilist_user_id", "source", "status", "n_entries", "added_at", "fetched_at", "error"]],
              OUTPUT_DIR / "users.csv")
    print(f"Assembly: {n_rows} ratings rows, {len(profiles)} user profiles, {len(metadata)} anime with metadata"
          + (" (partial)" if ASSEMBLE_PARTIAL else ""))
    return True


assembled = assemble(seed_state, load_usernames())

Assembly: 7485213 ratings rows, 9483 user profiles, 15000 anime with metadata


In [18]:
# Validation: output files
check = Check()
if not assembled:
    print("Not assembled yet")
else:
    ratings = pd.read_csv(OUTPUT_DIR / "ratings.csv.gz", dtype={"username": str}, keep_default_na=False, na_values={c: [""] for c in RATING_COLUMNS if c != "username"})
    profiles = pd.read_csv(OUTPUT_DIR / "user_profiles.csv", dtype={"user": str}, keep_default_na=False, na_values={"Days": [""], "Mean Score": [""], "Gender": [""]})
    metadata = pd.read_csv(OUTPUT_DIR / "anime_metadata.csv")
    users_out = pd.read_csv(OUTPUT_DIR / "users.csv", dtype=str, keep_default_na=False)
    usernames = load_usernames()
    done = usernames[usernames["status"] == "done"]
    max_score = SCORE_MAX[SCORE_FORMAT]

    check(not ratings.duplicated(["anilist_user_id", "anime_id"]).any(), f"ratings: {len(ratings)} rows, no duplicate user/anime pairs")
    check(ratings["score"].between(0, max_score).all(), f"scores within 0-{max_score} ({SCORE_FORMAT})")
    rated = ratings[ratings["is_rated"]]
    check(rated["rating"].between(10 / max_score, 10).all() and ratings.loc[~ratings["is_rated"], "rating"].isna().all(),
          "rating is on the 1-10 scale for rated entries and empty for unrated ones")
    check(ratings[["anilist_user_id", "anime_id", "username", "source"]].notna().all().all(), "no missing user or anime IDs, usernames or sources")
    check((not DROP_UNRATED) or ratings["is_rated"].all(), f"DROP_UNRATED = {DROP_UNRATED} respected")
    check(set(ratings["anilist_user_id"]) <= set(done["anilist_user_id"]), "only done users in ratings")
    check(len(profiles) == len(done) and not profiles["anilist_user_id"].duplicated().any(), f"user_profiles: one row per done user ({len(profiles)})")
    check(profiles["Gender"].isna().all(), "Gender left empty (not available from AniList)")
    check(len(users_out) == len(usernames), f"users.csv lists all {len(usernames)} usernames")
    manual_done = set(done.loc[done["source"] == "manual", "anilist_user_id"])
    check(manual_done <= set(profiles["anilist_user_id"]), f"{len(manual_done)} fetched manual user(s) present in the outputs")
    covered = rated["anime_id"].isin(metadata["anime_ID"]).mean() if len(rated) else 0
    print(f"{covered:.1%} of rated entries are for anime with metadata")
    display(hide_usernames(ratings).head())
    display(hide_usernames(profiles).head())
check.done()

[PASS] ratings: 7485213 rows, no duplicate user/anime pairs
[PASS] scores within 0-100 (POINT_100)
[PASS] rating is on the 1-10 scale for rated entries and empty for unrated ones
[PASS] no missing user or anime IDs, usernames or sources
[PASS] DROP_UNRATED = False respected
[PASS] only done users in ratings
[PASS] user_profiles: one row per done user (9483)
[PASS] Gender left empty (not available from AniList)
[PASS] users.csv lists all 10001 usernames
[PASS] 1 fetched manual user(s) present in the outputs
99.5% of rated entries are for anime with metadata


,source,anime_id,anime_title,status,score,rating,is_rated,progress,repeat,started_at,completed_at,updated_at,created_at
0,seed,6682,11eyes,COMPLETED,60,6.0,True,12,0,NaN,NaN,NaN,NaN
1,seed,21366,3-gatsu no Lion,COMPLETED,100,10.0,True,22,0,NaN,NaN,NaN,NaN
2,seed,98478,3-gatsu no Lion 2nd Season,COMPLETED,100,10.0,True,22,0,NaN,NaN,NaN,NaN
3,seed,100526,3D Kanojo: Real Girl,COMPLETED,70,7.0,True,12,0,NaN,2018-06-21,2019-03-15T12:34:08+00:00,NaN
4,seed,99725,A.I.C.O. -Incarnation-,COMPLETED,80,8.0,True,12,0,NaN,NaN,NaN,NaN


,Gender,Watching,Completed,On-Hold,Dropped,Plan to Watch,Total Entries,Rewatched,Episodes,Days,Mean Score,source
0,NaN,6,1386,0,0,134,1526,0,23675,384.5,7.14,seed
1,NaN,8,1019,17,180,317,1541,61,9778,165.1,8.33,seed
2,NaN,7,158,49,48,90,352,17,3117,56.5,7.28,seed
3,NaN,3,300,9,41,40,393,41,4996,97.8,7.06,seed
4,NaN,8,413,5,32,149,607,93,6463,113.7,8.01,seed


## Summary

In [19]:
usernames = load_usernames()
status = usernames["status"].value_counts()
n_seed, n_manual = int((usernames["source"] == "seed").sum()), int((usernames["source"] == "manual").sum())
print(f"Unique users listed:   {len(usernames)}")
print(f"  seed / manual:       {n_seed} / {n_manual}")
print(f"  lists fetched:       {int(status.get('done', 0))}")
print(f"  skipped, private:    {int(status.get('private', 0))}")
print(f"  skipped, not found:  {int(status.get('not_found', 0))}")
print(f"  failed (retryable):  {int(status.get('failed', 0))}")
print(f"  still pending:       {int(status.get('pending', 0))}")

if (OUTPUT_DIR / "ratings.csv.gz").exists():
    is_rated = pd.read_csv(OUTPUT_DIR / "ratings.csv.gz", usecols=["is_rated"])["is_rated"]
    print(f"Total ratings rows:    {len(is_rated)} ({int(is_rated.sum())} rated, {int((~is_rated).sum())} unrated"
          + (", unrated dropped" if DROP_UNRATED else "") + ")")
print(f"Anime with metadata:   {len(load_metadata_progress()['done'])}")

mal = load_mal_synopses()
if len(mal):
    mal_status = mal["status"].value_counts()
    print(f"MAL synopses:          {int(mal_status.get('found', 0))} found, {int(mal_status.get('empty', 0))} empty on MAL, "
          f"{int(mal_status.get('not_found', 0))} not on MAL")
mal_part = f", plus {REQUEST_STATS['mal_requests']} to {mal_source()}" if REQUEST_STATS["mal_requests"] else ""
print(f"Requests this session: {REQUEST_STATS['requests']} ({REQUEST_STATS['retries']} retries, {REQUEST_STATS['rate_limited']} rate limited){mal_part}")

print("\nFiles:")
for path in sorted(p for p in OUTPUT_DIR.rglob("*") if p.is_file() and ".tmp_" not in p.name):
    print(f"  {path.relative_to(OUTPUT_DIR).as_posix():<45}{path.stat().st_size / 1024:10.1f} KB")

Unique users listed:   10001
  seed / manual:       10000 / 1
  lists fetched:       9483
  skipped, private:    518
  skipped, not found:  0
  failed (retryable):  0
  still pending:       0
Total ratings rows:    7485213 (4557327 rated, 2927886 unrated)
Anime with metadata:   15000
MAL synopses:          14339 found, 290 empty on MAL, 31 not on MAL
Requests this session: 0 (0 retries, 0 rate limited)

Files:
  .gitkeep                                            0.0 KB
  anime_metadata.csv                              20925.5 KB
  checkpoints/mal_synopsis_parts/part_00001.csv       6.6 KB
  checkpoints/mal_synopsis_parts/part_00002.csv      90.4 KB
  checkpoints/mal_synopsis_parts/part_00003.csv      79.3 KB
  checkpoints/mal_synopsis_parts/part_00004.csv      73.1 KB
  checkpoints/mal_synopsis_parts/part_00005.csv      71.5 KB
  checkpoints/mal_synopsis_parts/part_00006.csv      71.4 KB
  checkpoints/mal_synopsis_parts/part_00007.csv      64.3 KB
  checkpoints/mal_synopsis_parts/part